# Importations nécessaire

In [42]:
import numpy as np
from gurobipy import *
import pandas as pd
from collections import defaultdict

# Préparation des données
Récuperation des matrices de gains, de coût, des listes des bâtiments et de leurs types (même code que le jalon précédent)

In [43]:
# --- 1. Lecture du fichier ---
df = pd.read_excel("efficacity_data_v1.xlsx")

# ============================
# PARTIE A — CALIBRATED
# ============================

df_calibrated = (
    df[df["id_simulation"].astype(str).str.contains("calibrated", case=False)]
    [["building_name", "building_type", "id_simulation", "conso_total_mwh_an"]]
)

# Comptage par catégorie
nb_buildings_by_type = (
    df_calibrated
    .groupby("building_type")["building_name"]
    .count()
    .to_dict()
)

# Liste des calibrated par bâtiment
calibrated_list = (
    df_calibrated
    .groupby("building_name")["conso_total_mwh_an"]
    .apply(list)
    .to_dict()
)

# ============================
# PARTIE B — SANS CALIBRATED
# ============================
# --- sANS les calibrated ---
df_nc = df[
    ~df["id_simulation"].astype(str).str.contains("calibrated", case=False)
]

# --- 2. Groupement ---
group_gains = df_nc.groupby("building_name")["gains_totaux_mwh_an"].apply(list)
group_cost  = df_nc.groupby("building_name")["cout_investissement_euros"].apply(list)

# --- 3. Taille maximale ---
max_len = max(
    group_gains.apply(len).max(),
    group_cost.apply(len).max()
)

# --- 4. Matrices à taille fixe ---
gains_matrix = np.array([
    vals + [np.inf] * (max_len - len(vals))
    for vals in group_gains
])

cost_matrix = np.array([
    vals + [np.inf] * (max_len - len(vals))
    for vals in group_cost
])

group_type = (
    df_nc
    .groupby("building_name")["building_type"]
    .first()
)
# --- 5. Métadonnées utiles ---
building_names = group_gains.index.tolist()
building_types = group_type.loc[building_names].tolist()

grouped = df_nc.groupby("building_name", sort=False)["id_simulation"].apply(list)
renovation_names_per_building = grouped.to_dict()

df_calibrated = df_calibrated[
    df_calibrated["building_name"].isin(building_names)
]

## Nouveauté: Construction du graphe de transitions

Pour un bâtiment b, on construit l'espace des transitions de la rénovation r vers la rénovation r' ssi $r\subset r'$

Alors la transition $r \rightarrow r'$ aura pour gain: gain(r') - gain(r) et pour coût: coût(r')-coût(r)

On initialise aussi des transitions intiales $None \rightarrow r$

In [44]:
# ============================
# PARTIE C — TRANSITION GRAPH
# ============================

transition_graph = {}
# Colonnes travaux (G → N)
renov_cols = df.columns[6:14]
# --- 3. Construction par bâtiment ---
for building, group in df_nc.groupby("building_name", sort=False):

    group = group.reset_index(drop=True)

    works = group[renov_cols].astype(bool).values
    costs = group["cout_investissement_euros"].values
    gains = group["gains_totaux_mwh_an"].values
    ids = group["id_simulation"].tolist()

    n = len(group)

    # ========================
    # MATRICE COMPATIBILITE
    # ========================

    compat = np.zeros((n, n), dtype=int)

    for i in range(n):
        for j in range(n):

            if np.all(~works[j] | works[i]):
                compat[i, j] = 1

    # ========================
    # GENERATION DES TRANSITIONS
    # ========================

    transitions = []

    # état initial (aucun travaux)
    for i in range(n):

        transitions.append({
            "from": "none",
            "to": ids[i],
            "cost": costs[i],
            "gain": gains[i]
        })

    # transitions entre rénovations
    for i in range(n):
        for j in range(n):

            if i == j:
                continue

            # j ⊂ i
            if compat[i, j] == 1 and compat[j, i] == 0:

                transitions.append({
                    "from": ids[j],
                    "to": ids[i],
                    "cost": costs[i] - costs[j],
                    "gain": gains[i] - gains[j]
                })

    transition_graph[building] = transitions

edges = {}

for b in building_names:
    edges[b] = list(range(len(transition_graph[b])))

cost = {}
gain = {}
from_state = {}
to_state = {}

for b in building_names:
    for e, edge in enumerate(transition_graph[b]):

        cost[b, e] = edge["cost"]
        gain[b, e] = edge["gain"]
        from_state[b, e] = edge["from"]
        to_state[b, e] = edge["to"]


# ============================
# Isolation des travaux spécifiques (utile pour l'excel)
works_by_sim = {}

for _, row in df_nc.iterrows():

    works_by_sim[row["id_simulation"]] = {
        col: bool(row[col])
        for col in renov_cols
    }


In [45]:
print(transition_graph["abri_bouliste"])

[{'from': 'none', 'to': 'abri_bouliste_simulation_1', 'cost': 9352.936, 'gain': 2.896469999999999}, {'from': 'none', 'to': 'abri_bouliste_simulation_10', 'cost': 49690.17, 'gain': 4.898935}, {'from': 'none', 'to': 'abri_bouliste_simulation_11', 'cost': 62094.12, 'gain': 6.087382999999999}, {'from': 'none', 'to': 'abri_bouliste_simulation_12', 'cost': 65558.17, 'gain': 6.060181999999999}, {'from': 'none', 'to': 'abri_bouliste_simulation_13', 'cost': 34850.99446540134, 'gain': 3.130339999999999}, {'from': 'none', 'to': 'abri_bouliste_simulation_14', 'cost': 59946.41, 'gain': 5.379875999999999}, {'from': 'none', 'to': 'abri_bouliste_simulation_15', 'cost': 50441.98, 'gain': 4.499982999999999}, {'from': 'none', 'to': 'abri_bouliste_simulation_2', 'cost': 36596.06, 'gain': 5.703981999999999}, {'from': 'none', 'to': 'abri_bouliste_simulation_3', 'cost': 19609.17, 'gain': 3.04185}, {'from': 'none', 'to': 'abri_bouliste_simulation_4', 'cost': 46852.3, 'gain': 6.000639}, {'from': 'none', 'to': 

In [46]:
# --- Exemple ---
example_building = list(transition_graph.keys())[3]

print("Building:", example_building)
for t in transition_graph[example_building][10:15]:
    print(t)

Building: ateliers_municipaux_(ctm)
{'from': 'ateliers_municipaux_(ctm)_simulation_5', 'to': 'ateliers_municipaux_(ctm)_simulation_1', 'cost': 334977.17999999993, 'gain': 198.3044188829827}
{'from': 'ateliers_municipaux_(ctm)_simulation_1', 'to': 'ateliers_municipaux_(ctm)_simulation_10', 'cost': 379793.97, 'gain': 71.61618698392931}
{'from': 'ateliers_municipaux_(ctm)_simulation_4', 'to': 'ateliers_municipaux_(ctm)_simulation_10', 'cost': 353521.25, 'gain': 254.52512333211652}
{'from': 'ateliers_municipaux_(ctm)_simulation_5', 'to': 'ateliers_municipaux_(ctm)_simulation_10', 'cost': 714771.1499999999, 'gain': 269.920605866912}
{'from': 'ateliers_municipaux_(ctm)_simulation_6', 'to': 'ateliers_municipaux_(ctm)_simulation_10', 'cost': 1420130.14, 'gain': 397.67630080728065}


In [47]:
Conso_total = 0.9 * np.sum(df_calibrated["conso_total_mwh_an"]) # 10% pour la sobriété énergétique
nbr_annees = 2050 - 2026 + 1
nbr_batiments = gains_matrix.shape[0]
#nbr_renovations_max = gains_matrix.shape[1] (n'est plus utile)
print("Conso total:", Conso_total, "MWh/an")
print("Nombre de bâtiments:", nbr_batiments)

Conso total: 18759.129691178903 MWh/an
Nombre de bâtiments: 71


Les variables de décisions sont:
$$ y_{b,e,t} = 1 \quad \text{Si l'arrête e (d'un état r à r') est réalisée sur le bâtiment b à l'année t, 0 sinon}

In [48]:
m = Model("Efficacity Optimization")
y = {}

for b in building_names:
    for e in edges[b]:
        for t in range(nbr_annees):

            y[b, e, t] = m.addVar(
                vtype=GRB.BINARY,
                name=f"y_{b}_{e}_{t}"
            )
m.update()

Contrainte de l'unicité de transitions par bâtiment par an
$$ \forall b \quad \forall t \quad \sum_e y_{b,e,t} \leq 1

In [49]:
for b in building_names:
    for t in range(nbr_annees):
        m.addConstr(
            quicksum(
                y[b, e, t] for e in edges[b]
            ) <= 1,
            name=f"one_transition_{b}_{t}"
        )

Contrainte de l'unicité d'une rénovation
$$ \forall b \quad \forall r \quad \sum_t \sum_{e,to(e)=r}y_{b,e,t} \leq 1

In [50]:
for b in building_names:

    states = set(edge["to"] for edge in transition_graph[b])

    for r in states:

        incoming = [
            e for e, edge in enumerate(transition_graph[b])
            if edge["to"] == r
        ]

        m.addConstr(
            quicksum(
                y[b,e,t]
                for e in incoming
                for t in range(nbr_annees)
            ) <= 1,
            name=f"state_once_{b}_{r}"
        )

Contrainte: Quitter None une seule fois
$$ \sum_t\sum_{e,from(e)=None}y_{b,e,t} \leq 1

In [51]:
for b in building_names:

    none_edges = [
        e for e, edge in enumerate(transition_graph[b])
        if edge["from"] == "none"
    ]

    m.addConstr(
        quicksum(
            y[b,e,t]
            for e in none_edges
            for t in range(nbr_annees)
        ) <= 1,
        name=f"leave_none_once_{b}"
    )

Contrainte de budget annuel:
$$ \forall t \quad \sum_{b,e} y_{b,e,t}*cout_{b,e} \leq 2M€ 

In [52]:
for t in range(nbr_annees):
    m.addConstr(
        quicksum(
            cost[b, e] * y[b, e, t]
            for b in building_names
            for e in edges[b]
        ) <= 2e6,
        name=f"budget_{t}"
    )

Contrainte: Disponibilité par catégorie
$$ \forall cat \quad \forall t \quad \sum_{b \in cat} \sum_e y_{b,e,t}*\frac{1}{card(cat)} \leq 0.2

In [53]:
buildings_by_cat = defaultdict(list)

for b_name, cat in zip(building_names, building_types):
    buildings_by_cat[cat].append(b_name)

# contraintes de disponibilité
for cat, b_names in buildings_by_cat.items():

    card_cat = len(b_names)

    for t in range(nbr_annees):

        m.addConstr(
            quicksum(
                y[b, e, t]
                for b in b_names
                for e in edges[b]
            ) <= 0.2 * card_cat,
            name=f"availability_{cat}_{t}"
        )

Contrainte de cohérence des flux
$$ \forall b \quad \forall r \quad \forall t \quad \sum_{\tau \leq t}\sum_{e:to(e)=r}y_{b,e,\tau}\geq \sum_{\tau \leq t}\sum_{e:from(e)=r}y_{b,e,\tau}

In [54]:
for b in building_names:

    states = set()

    for edge in transition_graph[b]:
        states.add(edge["from"])
        states.add(edge["to"])

    states.discard("none")

    for r in states:

        incoming = [
            e for e, edge in enumerate(transition_graph[b])
            if edge["to"] == r
        ]

        outgoing = [
            e for e, edge in enumerate(transition_graph[b])
            if edge["from"] == r
        ]

        for t in range(nbr_annees):

            m.addConstr(
                quicksum(
                    y[b,e,tau] for e in incoming for tau in range(t+1)
                )
                >=
                quicksum(
                    y[b,e,tau] for e in outgoing for tau in range(t+1)
                ),
                name=f"flow_{b}_{r}_{t}"
            )

Fonction Objectif: minimiser les écarts par rapport aux objectifs
$$ min((0.4*Conso_0 - \sum_{t=0}^{t=4}\sum_{b,e}y_{b,e,t}*gain_{b,e}) + \\
(0.5*Conso_0 - \sum_{t=0}^{t=14}\sum_{b,e}y_{b,e,t}*gain_{b,e}) + \\
(0.6*Conso_0 - \sum_{t=0}^{t=24}\sum_{b,e}y_{b,e,t}*gain_{b,e}))

In [55]:
m.setObjective(0.25*(0.4*Conso_total-quicksum(y[b, e, t]* gain[b, e]
    for b in building_names
    for e in edges[b]
    for t in range(0, 5)))
    + 0.25*(0.5*Conso_total-quicksum(y[b, e, t]* gain[b, e]
    for b in building_names
    for e in edges[b]
    for t in range(0, 15)))
    + 0.5*(0.6*Conso_total-quicksum(y[b, e, t]* gain[b, e]
    for b in building_names
    for e in edges[b]
    for t in range(0, 25)))
    , GRB.MINIMIZE)

In [56]:
#Fonction objectif alternative pour éventuellement tester la maximisation de gains en 2050
""" m.setObjective(quicksum(y[b, e, t] * gain[b, e] for b in building_names for e in edges[b] for t in range(25)), GRB.MAXIMIZE) """

' m.setObjective(quicksum(y[b, e, t] * gain[b, e] for b in building_names for e in edges[b] for t in range(25)), GRB.MAXIMIZE) '

Résolution du modèle

In [57]:
#Remarque: Time Limit nécessaire sinon le modèle ne termine pas (+10min !!!)
#Mais au delà de 2min le gain n'est plus significatif
m.setParam('TimeLimit', 2*60)
m.update()
m.optimize()


Set parameter TimeLimit to value 120
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i5-1135G7 @ 2.40GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  120

Optimize a model with 22381 rows, 62625 columns and 1643250 nonzeros (Min)
Model fingerprint: 0xed779dbf
Model has 62525 linear objective coefficients and an objective constant of 9.8485430878689240e+03
Variable types: 0 continuous, 62625 integer (62625 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+06]
  Objective range  [5e-06, 8e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+06]

Found heuristic solution: objective 3628.1540479
Found heuristic solution: objective 3617.7354276
Presolve removed 5713 rows and 2649 columns (presolve time = 5s)...
Presolve removed 6175 rows and 3437 columns (presolve time = 10s)...
Presol

Fonction objectif obtenue (Ecart total par rapport aux objectifs du décret tertiaire)

In [58]:
print(m.ObjVal)

1886.1442004609635


Affichage des objectifs réalisés avec cette solution

In [59]:
obj_2030 = sum(
    y[b, e, t].X * gain[b, e]
    for b in building_names
    for e in edges[b]
    for t in range(0, 5)
) / Conso_total


obj_2040 = sum(
    y[b, e, t].X * gain[b, e]
    for b in building_names
    for e in edges[b]
    for t in range(0, 15)
) / Conso_total


obj_2050 = sum(
    y[b, e, t].X * gain[b, e]
    for b in building_names
    for e in edges[b]
    for t in range(nbr_annees)
) / Conso_total


print(f"Objectif 2030 : {obj_2030:.4f} (cible 0.4)")
print(f"Objectif 2040 : {obj_2040:.4f} (cible 0.5)")
print(f"Objectif 2050 : {obj_2050:.4f} (cible 0.6)")

Objectif 2030 : 0.2963 (cible 0.4)
Objectif 2040 : 0.4493 (cible 0.5)
Objectif 2050 : 0.4762 (cible 0.6)


N.B: En retirant la contrainte du budget, on atteint 50% de gain en 2050 (en ayant pour objectif de le maximiser à cet horizon)

Ce qui suit est la cosntruction du document excel plan_renovation_solution.xslx à consulter dans le même dossier.

(Le code ne doit pas être réexcuté au risque de supprimer les graphes inserés manuellement)

In [60]:
def works_added(from_state, to_state):
    """ Isolation des travaux spécifiques à partir d'une arrête du graphe
    Utile pour connaître exactement quelles rénovations sont faites"""
    if from_state == "none":
        return [
            col for col, v in works_by_sim[to_state].items() if v
        ]

    return [
        col
        for col in renov_cols
        if works_by_sim[to_state][col] and not works_by_sim[from_state][col]
    ]

In [61]:
""" rows = []

years = list(range(2026, 2051))

for (b, e, t), var in y.items():

    if var.X > 0.5:

        tr = transition_graph[b][e]

        added = works_added(tr["from"], tr["to"])

        rows.append({
            "building": b,
            "year": t+2026,
            "from_state": tr["from"],
            "to_state": tr["to"],
            "works_added": ", ".join(added),
            "gain": tr["gain"],
            "cost": tr["cost"]
        })

df_solution = pd.DataFrame(rows)

df_solution = df_solution.sort_values(["building","year"])


# =========================
# tableau bâtiment × année
# =========================

timeline = (
    df_solution
    .pivot(index="building", columns="year", values="to_state")
    .reindex(columns=years)
)

# ==========================
# Travaux par année par bâtiment
# ==========================

timeline_works = (
    df_solution
    .pivot(index="building", columns="year", values="works_added")
    .reindex(columns=years)
)

# =========================
# stats par année
# =========================

stats = (
    df_solution
    .groupby("year")
    .agg(
        renovations=("building","count"),
        gain_total=("gain","sum"),
        cost_total=("cost","sum")
    )
    .reindex(years, fill_value=0)
    .reset_index()
)
## =========================
# Disponibilité par catégorie
# =========================

# mapping bâtiment -> catégorie
building_type_map = dict(zip(building_names, building_types))

df_solution["category"] = df_solution["building"].map(building_type_map)

years = list(range(2026, 2026 + nbr_annees))
categories = list(nb_buildings_by_type.keys())

# nombre de rénovations par catégorie et par année
availability = (
    df_solution
    .groupby(["category", "year"])
    .size()
    .reindex(
        pd.MultiIndex.from_product([categories, years],
        names=["category","year"]),
        fill_value=0
    )
    .reset_index(name="renovations_done")
)

# capacité maximale autorisée
availability["max_allowed"] = availability["category"].map(
    lambda c: 0.2 * nb_buildings_by_type[c]
)

# ratio d'utilisation
availability["usage_ratio"] = (
    0.2*availability["renovations_done"] / availability["max_allowed"]
)


# =========================
# Export Excel
# =========================

with pd.ExcelWriter("plan_renovation_solution.xlsx") as writer:

    df_solution.to_excel(writer, sheet_name="transitions", index=False)
    timeline.to_excel(writer, sheet_name="timeline_renovations")
    timeline_works.to_excel(writer, sheet_name="timeline_travaux")
    stats.to_excel(writer, sheet_name="stats_annees", index=False)
    availability.to_excel(writer, sheet_name="availability_by_category", index=False) """

' rows = []\n\nyears = list(range(2026, 2051))\n\nfor (b, e, t), var in y.items():\n\n    if var.X > 0.5:\n\n        tr = transition_graph[b][e]\n\n        added = works_added(tr["from"], tr["to"])\n\n        rows.append({\n            "building": b,\n            "year": t+2026,\n            "from_state": tr["from"],\n            "to_state": tr["to"],\n            "works_added": ", ".join(added),\n            "gain": tr["gain"],\n            "cost": tr["cost"]\n        })\n\ndf_solution = pd.DataFrame(rows)\n\ndf_solution = df_solution.sort_values(["building","year"])\n\n\n# =========================\n# tableau bâtiment × année\n# =========================\n\ntimeline = (\n    df_solution\n    .pivot(index="building", columns="year", values="to_state")\n    .reindex(columns=years)\n)\n\n# ==========================\n# Travaux par année par bâtiment\n# ==========================\n\ntimeline_works = (\n    df_solution\n    .pivot(index="building", columns="year", values="works_added")\n 